# 02.5 V2 — WDC Download Review

Review the WDC candidate URLs collected by `02_v2_wdc_download.ipynb`.
Check what types downloaded, property count distributions, and domain diversity.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse
from io import BytesIO
from IPython.display import Image as IPImage

WDC_PATH    = Path('../data/raw/wdc_candidate_urls.jsonl')
MERGED_PATH = Path('../data/raw/all_candidate_urls.jsonl')

wdc_records = []
if WDC_PATH.exists():
    with open(WDC_PATH) as f:
        for line in f:
            wdc_records.append(json.loads(line))
    print(f'WDC candidates: {len(wdc_records):,}')
else:
    print(f'Not found: {WDC_PATH}  — run 02_v2_wdc_download.ipynb first')

merged_records = []
if MERGED_PATH.exists():
    with open(MERGED_PATH) as f:
        for line in f:
            merged_records.append(json.loads(line))
    print(f'Merged pool:     {len(merged_records):,}')

In [ ]:
if not wdc_records:
    print('No data yet — check back when 02_v2 finishes')
else:
    type_counts   = Counter(r['schema_type'] for r in wdc_records)
    prop_counts   = [int(r.get('property_count', 0)) for r in wdc_records]

    def get_tld(url):
        try:
            host = urlparse(url).netloc.lower().lstrip('www.')
            parts = host.split('.')
            if len(parts) >= 3 and parts[-2] in ('co', 'com', 'org', 'net'):
                return '.'.join(parts[-2:])
            return parts[-1]
        except Exception:
            return 'unknown'

    tld_counts = Counter(get_tld(r['url']) for r in wdc_records)

    print(f'Types downloaded: {len(type_counts)}')
    print(f'Avg properties:   {np.mean(prop_counts):.1f}')
    print(f'Median properties: {np.median(prop_counts):.1f}')
    print()
    print('TYPE BREAKDOWN')
    for t, n in type_counts.most_common():
        print(f'  {t:25s} {n:6,}')
    print()
    print('TOP TLDs')
    for tld, n in tld_counts.most_common(10):
        print(f'  .{tld:15s} {n:6,}')

In [ ]:
if wdc_records:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('WDC Candidate URLs — 02_v2 Output', fontsize=13, fontweight='bold')

    # Type bar chart
    ax = axes[0]
    types, tcounts = zip(*type_counts.most_common())
    ax.barh(range(len(types)), tcounts, color='steelblue')
    ax.set_yticks(range(len(types)))
    ax.set_yticklabels(types, fontsize=9)
    ax.set_xlabel('Count')
    ax.set_title('Schema type distribution')
    ax.invert_yaxis()

    # Property count histogram
    ax = axes[1]
    ax.hist(prop_counts, bins=30, color='steelblue', edgecolor='white', range=(0, 50))
    ax.set_xlabel('Property count')
    ax.set_ylabel('Pages')
    ax.set_title('Property count distribution')
    ax.axvline(np.median(prop_counts), color='red', linestyle='--', label=f'Median {np.median(prop_counts):.0f}')
    ax.legend(fontsize=8)

    # TLD pie
    ax = axes[2]
    top_tlds = tld_counts.most_common(8)
    tld_labels, tld_vals = zip(*top_tlds)
    ax.pie(tld_vals, labels=[f'.{t}' for t in tld_labels], autopct='%1.0f%%', startangle=90)
    ax.set_title('TLD breakdown')

    plt.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    plt.close()
    buf.seek(0)
    display(IPImage(data=buf.getvalue()))

In [ ]:
# Merged pool summary (WDC + CC combined)
if merged_records:
    print(f'MERGED POOL: {len(merged_records):,} total URLs')
    print()
    print('Source breakdown:')
    for src, n in Counter(r.get('source', 'unknown') for r in merged_records).most_common():
        print(f'  {src:10s} {n:6,}  ({n/len(merged_records)*100:.0f}%)')
    print()
    print('Type breakdown (merged):')
    for t, n in Counter(r.get('schema_type', r.get('probable_type', '?')) for r in merged_records).most_common():
        print(f'  {t:25s} {n:6,}')
else:
    print('Merged pool not yet created — run 02_v2 merge cell after both 01_v2 and 02_v2 complete')